<a href="https://colab.research.google.com/github/Satyaharshith/25P35A1205_DriveReady_OPC_2026/blob/main/Day5/Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q langchain langchain-groq langgraph chromadb gspread pydantic

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

# Connect Google Sheets (Orders log)
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1u-1AcGkXVrC5aM-yg3g_ZozCbFgu68KhF2to8Y8uwPw"
try:
    sh = gc.open_by_url(SHEET_URL)
except Exception:
    sh = gc.create("Sweet Crumbs Orders")
    sh.sheet1.append_row(["name", "item", "qty", "date", "eggless", "total"])

ws = sh.sheet1
print("LLM & Sheet ready:", sh.url)

ImportError: cannot import name 'ModelRefusalError' from 'langchain_core.exceptions' (/usr/local/lib/python3.13/dist-packages/langchain_core/exceptions.py)

In [ ]:
import chromadb

FAQS = {
    "delivery":     "We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.",
    "payment":      "We accept UPI and cash. Orders above Rs 1,000 need a 50% advance by UPI.",
    "cancellation": "Cancellation is free up to 12 hours before pickup. After that, the advance is not refunded.",
    "photo_cake":   "Photo cakes and custom designs need 48 hours' notice.",
    "notice":       "Regular orders need 24 hours' notice.",
    "eggless":      "All our cakes have eggless versions at the same price.",
    "allergens":    "Plum cake contains nuts and dried fruit. Cupcakes and chocolate cake are nut-free.",
    "storage":      "Keep cakes in the fridge and eat them within 3 days.",
    "message":      "A message on the cake is free, up to 20 letters.",
    "candles":      "Candles are free with every cake.",
    "discounts":    "Bulk-order discounts need the owner's approval.",
    "pickup":       "Pickup is during opening hours: Monday to Saturday 9 am to 8 pm, Sunday 10 am to 6 pm.",
}

client = chromadb.Client()
faq = client.get_or_create_collection(name="sweet_crumbs_faq")
faq.upsert(ids=list(FAQS.keys()), documents=list(FAQS.values()))

print(faq.count(), "facts stored in ChromaDB")
print("Sample semantic search:", faq.query(query_texts=["Can you send the cake to my house?"], n_results=2)["documents"][0])

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm",
}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

@tool
def search_faq(question: str) -> str:
    """Search Sweet Crumbs' policies and FAQs: delivery, payment, cancellation,
    photo cakes, allergens, storage, candles and messages on cakes.
    Use for any question about rules or policies, not for prices or opening hours."""
    results = faq.query(query_texts=[question], n_results=3)
    return "\n".join(results["documents"][0])

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
RULES:  Use get_price for prices and is_open for timings; never guess them.
        For any policy question, use search_faq and answer ONLY from what it returns.
        If the answer isn't there, say "I'll check with the owner."
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order, search_faq],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

def chat(thread_id: str, text: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
        return result["messages"][-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print("Ravi 1:", chat("ravi", "Hi, I'm Ravi. Do you deliver? My order is about Rs 1,300."))
print("Ravi 2:", chat("ravi", "Does the plum cake have nuts?"))
print("Sravani 1:", chat("sravani", "Can I cancel tomorrow morning if I pick up tomorrow evening?"))
print("Ravi 3 (Memory + RAG):", chat("ravi", "What's my name, and will I pay for delivery?"))
print("Sravani 2 (Isolation check):", chat("sravani", "What's my name?"))

In [ ]:
import csv
import unicodedata

with open("tests.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "must_include"])
    writer.writerows([
        ["How much are 2 chocolate cakes?", "1300"],
        ["Are you open on Sunday?", "10 am"],
        ["Do you deliver?", "5 km"],
        ["Is the plum cake nut-free?", "nuts"],
        ["How much notice for a photo cake?", "48"],
        ["Can I pay by UPI?", "UPI"],
        ["Do you sell biryani?", "don't sell biryani|check with the owner"],
    ])

def norm(s: str) -> str:
    return unicodedata.normalize("NFKC", s).lower()

passed = 0
with open("tests.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for i, row in enumerate(rows, 1):
    reply = chat(f"test_{i}", row["question"])
    expected_options = [opt.strip() for opt in row["must_include"].split("|")]
    ok = any(norm(opt) in norm(reply) for opt in expected_options)
    passed += int(ok)
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] Q: {row['question']} -> {reply}")

print(f"\nScore: {passed}/{len(rows)} ({passed/len(rows)*100:.0f}%)")

In [ ]:
!pip -q install pymupdf sentence-transformers faiss-cpu pytesseract pillow
!apt-get -qq install -y tesseract-ocr poppler-utils

In [ ]:

import os
import io
import pymupdf
import pytesseract
from PIL import Image
from google.colab import files

uploaded = files.upload()
if uploaded:
    pdf_path = next(iter(uploaded))
else:
    pdf_path = "The_Little_Lantern.pdf"
    doc = pymupdf.open()
    page = doc.new_page()
    story = (
        "The Little Lantern\n"
        "In a small village surrounded by hills, a boy named Arjun lived with his\n"
        "grandmother. Every evening, he watched her light a little lantern and place it\n"
        "outside their house. One day, he asked why she did this when everyone already\n"
        "had lights at home. She smiled and said, 'A small light can make a big difference\n"
        "to someone who has lost their way.' Arjun did not understand her words, but he\n"
        "continued helping her light the lantern every evening.\n"
        "One stormy night, the electricity went out across the village. Heavy rain covered\n"
        "the roads, and a traveler lost his way while searching for shelter. Arjun noticed a\n"
        "figure struggling in the darkness and quickly carried the lantern to the roadside.\n"
        "Following its gentle glow, the traveler reached their house safely. Arjun and his\n"
        "grandmother offered him a warm meal, dry clothes, and a place to rest until the\n"
        "storm passed.\n"
        "The next morning, the grateful traveler thanked Arjun before continuing his\n"
        "journey. Arjun finally understood what his grandmother had meant. He began\n"
        "encouraging his neighbors to keep lanterns outside their homes, and soon the\n"
        "village became a welcoming place for anyone traveling after sunset. From that\n"
        "day onward, Arjun believed that even the smallest act of kindness could bring\n"
        "hope to someone in need."
    )
    page.insert_text((50, 50), story, fontsize=11)
    doc.save(pdf_path)
    doc.close()
    print(f"Created local PDF fallback: {pdf_path}")

def read_pdf(path: str) -> str:
    all_text = []
    with pymupdf.open(path) as pdf:
        for page_number, page in enumerate(pdf):
            text = page.get_text("text").strip()
            if not text:
                print(f"Using OCR for page {page_number + 1}")
                pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2), alpha=False)
                image = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(image, lang="eng").strip()
            if text:
                all_text.append(text)
    return "\n\n".join(all_text)

def split_text(text: str, chunk_size: int = 600, overlap: int = 150) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size")
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

pdf_text = read_pdf(pdf_path)
chunks = split_text(pdf_text)

print("Total characters:", len(pdf_text))
print("Total chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])

In [ ]:
from sentence_transformers import SentenceTransformer
from groq import Groq
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS ready: {index.ntotal} vectors ({embeddings.shape[1]} dimensions)")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def retrieve_context(question: str, top_k: int = 3) -> str:
    q_vec = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    q_vec = np.asarray(q_vec, dtype="float32")
    _, indices = index.search(q_vec, min(top_k, len(chunks)))
    return "\n\n".join(chunks[i] for i in indices[0] if i >= 0)

def ask_pdf(question: str) -> str:
    context = retrieve_context(question)
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a PDF question-answering assistant. "
                    "Answer ONLY using the provided PDF context. "
                    "If the answer is not present, say 'The answer is not available in the PDF.' "
                    "Do not invent information. Keep answers clear and concise."
                ),
            },
            {
                "role": "user",
                "content": f"PDF CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nANSWER:",
            },
        ],
        temperature=0,
        max_tokens=300,
    )
    return response.choices[0].message.content

sample_questions = [
    "Who did Arjun live with?",
    "Why did Arjun's grandmother place a lantern outside their house every evening?",
    "What did Arjun and his grandmother offer the traveler?",
    "Who is Virat Kohli?",
]

for q in sample_questions:
    print(f"Q: {q}\nA: {ask_pdf(q)}\n")

In [ ]:
# Part 3: Your 30% — BrightPage Stationery Agent (Memory + ChromaDB RAG)
STATIONERY_FAQS = {
    "delivery": "We offer free delivery within 3 km of campus for orders above Rs 500; otherwise delivery is Rs 30.",
    "same_day": "Orders placed before 2 pm are delivered the same evening between 5 pm and 8 pm.",
    "payment": "We accept UPI, cash, and debit/credit cards. Bulk printing orders above Rs 2,000 require a 50% UPI advance.",
    "returns": "Unused notebooks, pens, and art supplies can be exchanged within 7 days with the original bill.",
    "custom_print": "Custom thesis binding and spiral printing take 24 hours; urgent 3-hour express binding costs Rs 50 extra.",
    "paper_types": "We stock 75 GSM, 80 GSM, and 100 GSM A4 copier paper in reams of 500 sheets.",
    "bulk_discount": "Orders of 20 or more notebooks or 10+ paper reams get a flat 10% student and faculty discount.",
    "hours": "The store is open Monday to Saturday from 8:30 am to 9 pm, and Sunday from 10 am to 5 pm.",
    "calculators": "Casio scientific calculators (fx-991ES Plus and fx-991CW) come with a 3-year manufacturer warranty.",
    "art_kits": "All watercolor sets, acrylic paints, and sketchpads are non-toxic and student-grade.",
    "id_card_lanyard": "Custom printed college ID lanyards require a minimum order of 25 pieces and 3 days' lead time.",
    "gift_wrap": "Complimentary gift wrapping is available for purchases above Rs 300.",
    "pen_refills": "We carry blue, black, and red gel and ballpoint refills for Parker, Pilot, and Classmate pens.",
    "cancellation": "Custom printing orders cannot be cancelled once printing has started; regular orders can be cancelled anytime before dispatch.",
    "storage_tip": "Store fountain pen inks and acrylic markers upright in a cool, dry place away from direct sunlight.",
    "special_orders": "Imported drafting tools or rare art supplies not in stock require owner approval and a 5-day order window.",
}

stationery_col = client.get_or_create_collection(name="brightpage_stationery_faq")
stationery_col.upsert(ids=list(STATIONERY_FAQS.keys()), documents=list(STATIONERY_FAQS.values()))
print(f"Loaded {stationery_col.count()} facts into BrightPage Stationery ChromaDB")

STATIONERY_PRICES = {"a4 ream": 280, "spiral notebook": 90, "scientific calculator": 1150}

@tool
def get_stationery_price(item: str, qty: int = 1) -> str:
    """Get the price in rupees for a quantity of one stationery item: 'a4 ream', 'spiral notebook', or 'scientific calculator'."""
    price = STATIONERY_PRICES.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not in the quick price list. Available: {', '.join(STATIONERY_PRICES)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def search_stationery_faq(question: str) -> str:
    """Search BrightPage Stationery's policies and FAQs: delivery, same-day dispatch, payment, returns,
    custom thesis/spiral printing, paper GSM, bulk discounts, hours, calculators, art kits, lanyards, gift wrap, refills, and cancellations."""
    results = stationery_col.query(query_texts=[question], n_results=3)
    return "\n".join(results["documents"][0])

STATIONERY_SYSTEM = """ROLE:   You are the helpful store assistant for BrightPage Stationery.
RULES:  Use get_stationery_price for item prices; never guess them.
        For any store policy, delivery, printing, or product question, use search_stationery_faq and answer ONLY from what it returns.
        If the answer is not in the retrieved facts, say "I'll check with the owner."
FORMAT: Under 60 words, friendly and concise."""

stationery_agent = create_agent(
    model=llm,
    tools=[get_stationery_price, search_stationery_faq],
    system_prompt=STATIONERY_SYSTEM,
    checkpointer=InMemorySaver(),
)

def chat_stationery(thread_id: str, text: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = stationery_agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
    return result["messages"][-1].content

print("\n--- Multi-Thread Memory + RAG Test ---")
print("Ananya 1:", chat_stationery("ananya", "Hi, I'm Ananya. How much for 2 A4 reams, and do you deliver free?"))
print("Karthik 1:", chat_stationery("karthik", "Hello! How long does custom thesis spiral printing take?"))
print("Ananya 2 (Memory):", chat_stationery("ananya", "What's my name and how many reams did I ask about?"))
print("Karthik 2 (Isolation):", chat_stationery("karthik", "What's my name?"))
print("Break-it test:", chat_stationery("ananya", "Do you sell laptops or mobile phones?"))

In [ ]:
with open("stationery_tests.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "must_include"])
    writer.writerows([
        ["How much are 3 spiral notebooks?", "270"],
        ["What is the minimum order for free delivery?", "500"],
        ["How many days do I have to exchange unused notebooks?", "7"],
        ["What is the bulk discount for 20 notebooks?", "10%"],
        ["How long is the warranty on Casio scientific calculators?", "3-year|3 year"],
        ["Do you repair broken smartphones?", "check with the owner|don't|do not"],
    ])

st_passed = 0
with open("stationery_tests.csv", encoding="utf-8") as f:
    st_rows = list(csv.DictReader(f))

for i, row in enumerate(st_rows, 1):
    reply = chat_stationery(f"st_test_{i}", row["question"])
    expected_options = [opt.strip() for opt in row["must_include"].split("|")]
    ok = any(norm(opt).replace("’", "'") in norm(reply).replace("’", "'") for opt in expected_options)
    st_passed += int(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] Q: {row['question']} -> {reply}")

print(f"\nYour 30% Pass Rate: {st_passed}/{len(st_rows)} ({st_passed/len(st_rows)*100:.0f}%)")